# 自注意力机制（Self-Attention / Multi-Head Attention）

自注意力机制（Self-Attention）是 Transformer 架构的核心组件，它允许模型在处理序列时，动态地为序列中的每个位置分配不同的注意力权重，从而捕捉全局依赖关系。下面我将从原理、计算过程到多头的扩展，逐步详细解释，并配合具体例子。

## 一、核心思想：为什么需要自注意力？

传统的 RNN/LSTM 是逐步处理序列的，长距离依赖信息容易被遗忘。而自注意力的核心思想是：
  - 序列中的每个元素，都能"看到"序列中的所有其他元素，并根据相关性动态决定关注哪些信息。
- 类比：你在读一句话时，大脑会自动将当前词与句子中其他相关词建立联系。例如读到"它"时，你会自动回溯找到它指代的是哪个名词。

## 二、自注意力的数学原理
### 2.1 三个关键向量：Query、Key、Value
对于输入序列中的每个词（或 token），模型通过三个不同的线性变换，生成三个向量：

| 向量            | 含义   | 类比           |
| ------------- | ---- | ------------ |
| **Query (Q)** | 查询向量 | "我在找什么信息？"   |
| **Key (K)**   | 键向量  | "我有什么信息？"    |
| **Value (V)** | 值向量  | "我的实际内容是什么？" |

### 2.2 注意力分数计算（Scaled Dot-Product Attention）

#### 公式：
- Attention(Q,K,V)=softmax( $\frac{QK^T}{\sqrt{d_k}}$)V 

计算步骤：
1.计算相似度：Q×$K_T$（点积），得到注意力分数矩阵
2.缩放：除以 $\sqrt{d_k}$（防止点积过大导致 softmax 梯度消失）
3.归一化：softmax，使每行和为 1
4.加权求和：用 softmax 结果对 V 加权，得到输出

## 三、具体例子详解
假设我们处理句子："猫 追 老鼠"
### 步骤 1：词嵌入
将每个词映射为向量（简化示例，假设维度 d=4 ）：

In [ ]:
猫   →  [1, 0, 1, 0]
追   →  [0, 1, 0, 1]  
老鼠 →  [1, 1, 0, 0]

### 步骤 2：生成 Q、K、V
通过可学习的权重矩阵 $W_Q,W_K,W_V$ 进行线性变换（为简化，假设权重矩阵为单位矩阵，即 Q=K=V=  输入）：

In [ ]:
Q = K = V = 
    [1, 0, 1, 0]   ← 猫
    [0, 1, 0, 1]   ← 追
    [1, 1, 0, 0]   ← 老鼠

### 步骤 3：计算注意力分数 Q×$K^T$

In [ ]:
        猫    追   老鼠
      ┌─────┬─────┬─────┐
  猫  │  2  │  1  │  1  │   ← 猫与自身的点积=2, 与追=1, 与老鼠=1
      ├─────┼─────┼─────┤
  追  │  1  │  2  │  1  │   ← 追与猫=1, 与自身=2, 与老鼠=1
      ├─────┼─────┼─────┤
老鼠  │  1  │  1  │  2  │   ← 老鼠与猫=1, 与追=1, 与自身=2
      └─────┴─────┴─────┘

### 步骤 4：缩放 + Softmax
假设 $d_k$=4 ，则  $\sqrt{d_k}$=2 。缩放后：

In [ ]:
        猫      追     老鼠
      ┌───────┬───────┬───────┐
  猫  │  1.0  │  0.5  │  0.5  │
      ├───────┼───────┼───────┤
  追  │  0.5  │  1.0  │  0.5  │
      ├───────┼───────┼───────┤
老鼠  │  0.5  │  0.5  │  1.0  │
      └───────┴───────┴───────┘

#### Softmax 后（每行和为 1）：

In [ ]:
        猫      追     老鼠
      ┌───────┬───────┬───────┐
  猫  │ 0.50  │ 0.25  │ 0.25  │  ← 猫最关注自己
      ├───────┼───────┼───────┤
  追  │ 0.25  │ 0.50  │ 0.25  │  ← 追最关注自己
      ├───────┼───────┼───────┤
老鼠  │ 0.25  │ 0.25  │ 0.50  │  ← 老鼠最关注自己
      └───────┴───────┴───────┘

### 步骤 5：加权求和得到输出
以"追"为例，输出 = 0.25×  猫的V + 0.50×  追的V + 0.25×  老鼠的V

In [ ]:
输出(追) = 0.25×[1,0,1,0] + 0.50×[0,1,0,1] + 0.25×[1,1,0,0]
        = [0.25+0+0.25, 0+0.50+0.25, 0.25+0+0, 0+0.50+0]
        = [0.50, 0.75, 0.25, 0.50]

这个输出向量融合了"追"自身的信息，以及它与"猫"和"老鼠"的关系信息。

## 四、多头注意力（Multi-Head Attention）
### 4.1 为什么需要多头？
单一的自注意力只能捕捉一种类型的关系。多头注意力通过并行运行多组独立的 Q/K/V 投影，让模型同时关注不同方面的信息：
- 头1：关注语法关系（主语-谓语-宾语）
- 头2：关注语义关系（同义词、反义词）
- 头3：关注位置关系（相邻词）
- ...
### 4.2 多头注意力的计算
- MultiHead(Q,K,V)=Concat($head_1,...,head_h$)$W^O$
  
其中每个头：
head_i =Attention($QW_i^Q,KW_i^K,VW_i^V$)

### 4.3 具体例子：多头如何分工
继续使用句子 "猫 追 老鼠"，假设有 2 个头：
#### 头 1（关注语法角色）：


In [ ]:
        猫      追     老鼠
  猫  │ 0.6   │ 0.2   │ 0.2   │  ← 猫关注自己（主语）
  追  │ 0.4   │ 0.3   │ 0.3   │  ← 追关注猫（谓语关注主语）
老鼠  │ 0.2   │ 0.4   │ 0.4   │  ← 老鼠关注追（宾语关注谓语）

#### 头 2（关注语义相似性）：

In [ ]:
        猫      追     老鼠
  猫  │ 0.5   │ 0.3   │ 0.2   │
  追  │ 0.3   │ 0.5   │ 0.2   │
老鼠  │ 0.2   │ 0.2   │ 0.6   │  ← 老鼠更关注自己（动物同类）

### 4.4 拼接与线性变换
将两个头的输出拼接，再通过一个线性层融合：

In [ ]:
头1输出(追) = [a1, a2, a3, a4]
头2输出(追) = [b1, b2, b3, b4]
拼接后     = [a1, a2, a3, a4, b1, b2, b3, b4]  (维度翻倍)
最终输出   = 线性层(拼接结果)  (投影回原始维度)

## 五、可视化总结

In [ ]:
输入序列: [猫] [追] [老鼠]
           ↓     ↓     ↓
        ┌─────────────────────┐
        │   多头注意力机制     │
        │  ┌───┐ ┌───┐ ┌───┐ │
        │  │头1│ │头2│ │头3│ │  ← 并行计算
        │  └───┘ └───┘ └───┘ │
        │      Concat        │
        │    Linear Proj     │
        └─────────────────────┘
           ↓     ↓     ↓
输出: [猫'] [追'] [老鼠']  ← 每个词都融入了全局上下文信息

## 六、关键优势总结

| 特性        | 说明                     |
| --------- | ---------------------- |
| **并行计算**  | 不像 RNN 需要逐步计算，所有位置同时处理 |
| **长距离依赖** | 任意两个位置直接交互，距离不影响效果     |
| **可解释性**  | 注意力权重矩阵直观展示模型"在看什么"    |
| **多头分工**  | 不同头学习不同类型的关系，增强表达能力    |


## 七、实际应用中的变体
- Masked Self-Attention（解码器）：在生成任务中，防止当前位置看到未来的信息
- Cross-Attention：Q 来自解码器，K/V 来自编码器，用于 Seq2Seq 任务
- Sparse Attention：降低计算复杂度，如 Longformer、BigBird
自注意力机制彻底改变了 NLP 领域，是 GPT、BERT、T5 等所有现代大语言模型的基石。理解它，就理解了 Transformer 的核心。